# Linear Regression with Time Series Data 📈

## Introduction

In Lesson 1 you extracted clean PM2.5 readings from MongoDB and saved them as a time-indexed DataFrame. That DataFrame is a **time series** — an ordered sequence of observations where the position of each row in time is not incidental, it is the whole point.

> ❓ What makes time series different from the house price and rental datasets in Projects 1 and 2?

### Why Time Series is Different

In Projects 1 and 2, the data was **cross-sectional**: each row represented an independent apartment or house. The order of rows was irrelevant. You could shuffle the DataFrame, split randomly, and still get valid training and test sets.

**Time series data violates every one of those assumptions:**

| Property | Cross-Sectional (Projects 1 & 2) | Time Series (Projects 3+) |
|---------|----------------------------------|--------------------------|
| Row independence | ✅ Rows are independent | ❌ Today depends on yesterday |
| Row order | Doesn't matter | Matters critically |
| Train/test split | Random shuffle is fine | Must respect chronological order |
| Cross-validation | K-fold is valid | Must use time-based folds |

In air quality data: today's PM2.5 level is correlated with yesterday's level, with last week's level, and — during a dust event — with readings from two weeks ago. The observations are **serially dependent**. Ignoring that dependence and treating the rows as i.i.d. would give you a model that looks good in cross-validation but fails completely in production, because it was allowed to see the future to predict the past.

> ❗️ **But wait — random splitting doesn't just violate a statistical assumption, it introduces data leakage.** If you randomly assign 20% of your rows to the test set, some test rows will be *earlier* in time than some training rows. The model trained on later data will "predict" earlier data — essentially predicting the past from the future. In deployment, that information isn't available. The test score is optimistic and meaningless.

### The Stationarity Assumption

AR and ARMA models (Lessons 3 and 4) formally require **stationarity**: a series is stationary if its statistical properties — mean, variance, autocorrelation structure — remain roughly constant over time.

> 💡 **Intuition:** a stationary series oscillates around a fixed mean with roughly constant spread. A non-stationary series might trend upward indefinitely, or explode in variance, or shift its average level after an event. Models trained on stationary data generalise better because the future has the same statistical character as the past.

For PM2.5 in Nairobi:
- **Within a season**: values fluctuate around a similar level → approximately stationary
- **Across seasons**: pollution shifts with weather and traffic patterns → possibly non-stationary

We will treat the data as approximately stationary for this lesson. If you were in production, you would test formally (e.g. ADF test) and difference the series if needed.

> ⚠️ **Caveat:** if a series has a strong trend (e.g., air quality is improving year-over-year due to regulations), a model trained on historical data will persistently overestimate future values. The trend makes the past and future statistically different — the core problem stationarity tries to solve.

### The Core Idea: Using History to Predict the Future

The simplest feature for predicting `y_t` (today's PM2.5) is `y_{t-1}` (yesterday's PM2.5). This is the **autoregressive insight**: the variable predicts itself. No external features needed; the history of the series carries predictive signal.

In this lesson you will build that intuition into a linear regression model using **lag features** — past values of the series lined up alongside the current values as input columns.

> 📌 **What you'll build today:**
> 1. A time series plot of PM2.5 to visualise the data before modelling
> 2. A lag feature (`lag_1`) and the feature matrix `X` / target vector `y`
> 3. A chronological 80/20 train/test split
> 4. A **persistence baseline** — the naivest honest forecast: "tomorrow = today"
> 5. A **linear regression model** trained on `lag_1`
> 6. Evaluation with MAE, RMSE, and R² — always compared against the baseline

➡️ Let's start by loading the data and visualising it. The `wrangle_data()` function from Lesson 1 gives us a clean, indexed PM2.5 DataFrame in a single call — that is the payoff of the reusable-function pattern.


## 1. Prepare Data

### Import

First, let’s import the clean data using the wrangling function we
created in Lesson 1.

**Code Task 3.2.1.1**: Import the `wrangle_data` function from
`mongo_wrangle` and call it to load the clean data into a DataFrame
named `df`. We will select records for the entire period of 2024.

> **Importing Your Module**
>
> Since we saved the wrangling function to `mongo_wrangle.py` in the
> previous lesson, we can now import it like any other Python library!

Below is an example. Make sure you get the exact JupyterHost id for this
session.

``` python
db = "air-quality"
collection = "nairobi"
host = "192.118.150.2"

# Load the data
df = wrangle_data(db, collection, host)
```

In [ ]:
import pandas as pd

# Import your wrangling function
from pymongo import MongoClient
from mongo_wrangle import wrangle_data
from load_mongo_data import load_nairobi_to_mongodb

db = "..."
collection = "..."
host = "..."

# load parquet to mongodb
load_nairobi_to_mongodb(host=host)


# Load the data
df = wrangle_data(..., ..., ...) # <-- make you pass the parameters in the exact order

print(f"Loaded {len(df):,} records")
print(f"Date range: {df.index.min()} to {df.index.max()}")
print("\nFirst 5 rows:")
df.head()

### Explore

Before fitting any model, visualise the raw data. A good time series plot tells you:
- Whether values trend upward or downward over time
- Whether there are regular daily or weekly cycles (periodicity)
- Whether there are sudden spikes or level shifts (shocks)
- How noisy the signal is relative to the pattern

> 🔍 **What to look for in the PM2.5 plot:**
> - **Daily cycles**: PM2.5 spikes during morning and evening rush hours (~6–9am and ~5–8pm); dips overnight
> - **Weekly patterns**: weekday pollution typically higher than weekend (less traffic)
> - **Spikes**: sudden large values (dust events, industrial incidents) that may challenge a simple linear model
> - **Overall level**: if the series drifts strongly upward or downward across the plotted period, you may need to address non-stationarity


**Code 3.2.1.1**: Create a time series plot showing the PM2.5 values
over time. Use `matplotlib` with explicit `fig` and `ax` objects.

In [ ]:
import matplotlib.pyplot as plt

# Create the plot
fig, ax = plt.subplots(figsize=(9, 6))

# Plot the time series
df.plot(ax=ax, y="pm25", legend=False)

ax.set_xlabel("Time")
ax.set_ylabel("PM2.5 Concentration")
ax.set_title("Nairobi PM2.5 Over Time")
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

> 📊 **Reading the plot:** the y-axis shows PM2.5 concentration in µg/m³. The WHO safe guideline is 15 µg/m³ (24-hour mean). Values above 35 µg/m³ are "unhealthy for sensitive groups". Notice how many hours in the dataset exceed these thresholds — that context motivates why accurate short-term forecasting matters for public health warnings.


### Create Lag Features

> 💡 **What is a lag feature?** A lag feature is the value of the time series at a previous time step, placed alongside the current value as a predictor. If the current row represents time `t`, the lag-1 feature contains the value from time `t-1`.
>
> ```
> t     | pm25 (target y) | lag_1 (feature X)
> 08:00 |      42         |  (NaN — no t-1)
> 09:00 |      45         |  42
> 10:00 |      38         |  45
> 11:00 |      50         |  38
> ```
>
> The first row always has a NaN in `lag_1` because there is no observation before it. Drop that row before modelling.

**Why does this work?** PM2.5 at 9am depends heavily on PM2.5 at 8am — not because time causes pollution, but because the same underlying conditions (traffic, weather, industrial activity) that elevated PM2.5 at 8am are still present at 9am. The lag feature captures that **persistence**: high values tend to be followed by high values.

> 📌 **The `shift(k)` method:** `series.shift(1)` moves all values down by 1 row. Row `i` in the shifted series contains the value that was originally in row `i-1`. Rows at the top become NaN because there's no previous value.


> **What Do You Notice?**
>
> Look at the plot carefully:
> 1. **Daily patterns?** You may notice PM2.5 spikes during rush hours (morning ~6–9am and evening ~5–8pm) due to traffic. Do you see values dip at night when traffic is lower?
> 2. **Weekly patterns?** PM2.5 might vary between weekdays (when traffic is heavy) and weekends (when traffic is lighter). Compare Monday–Friday pollution to Saturday–Sunday patterns.
> 3. **Weather effects?** Humidity and wind patterns (which vary seasonally) affect how pollution disperses. Dry seasons may show consistently higher levels.
>
> These temporal patterns — daily traffic cycles, weekly differences, seasonal trends — are what make time series unique and predictable!

**Code Task 3.2.1.3**: Create a single lag feature named `lag_1` that represents the PM2.5 value from 1 period ago. Drop rows with NaN values after creating the lag.


In [ ]:
# Create a copy of the dataframe
df_lags = (
    df
        
    # select data for only 2024
    .loc["2024"]
    # copy the original dataframe so we don't mess it up
    .copy()
    # Create single lag feature (1 period ago)
    .assign(lag_1=lambda x: x["pm25"].shift(...))
    # Drop rows with NaN values (from lag creation)
    .dropna()
    )

print(f"Shape after adding lag: {df_lags.shape}")
print("\nFirst 5 rows with lag feature:")
print(df_lags.head())

> 💡 **Building the feature matrix `X` and target vector `y`:**
>
> The sklearn convention requires:
> - `X` — a 2D DataFrame of shape `(n_samples, n_features)`. Even with one lag feature, use `df[["lag_1"]]` (double brackets) to get a DataFrame, not a Series.
> - `y` — a 1D Series of shape `(n_samples,)` with the target values (`pm25`).
>
> This single-lag setup is the simplest autoregressive feature matrix. In Lesson 3 we'll extend `X` to include multiple lags (lag_1, lag_2, ..., lag_p) automatically — that's the AR(p) model.


**Code Task 3.2.1.4**: Create the feature matrix `X` (using the `lag_1`
column) and target vector `y` (the `pm25` column).

> **Feature Matrix vs Target Vector**
>
> Remember: The Feature Matrix (X) must be two-dimensional (DataFrame),
> while the Target Vector (y) is one-dimensional (Series).

In [ ]:
# Create feature matrix X (lag_1 column)
X = df_lags[[...]]

# Create target vector y (pm25 column)
y = df_lags[...]

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print("\nFirst 5 rows of X:")
print(X.head())

### Linear Regression vs Autoregressive (AR) Models: A Bridge

At first glance, what we're doing here looks just like regular linear regression: we have a feature (`lag_1`) and predict a target (`pm25`). And it is linear regression — but applied to a time-structured problem. Here's the subtle but important distinction between this approach and the **AR models** we'll build in Lesson 3.

| Aspect | Our approach (Linear Regression on lags) | AR model (Lesson 3) |
|--------|------------------------------------------|---------------------|
| **Model** | Linear regression with `lag_1` as a feature | `AutoReg(p)` from statsmodels |
| **Equation** | `y_hat_t = beta_0 + beta_1 * y_{t-1}` | `y_t = c + phi_1*y_{t-1} + ... + phi_p*y_{t-p} + epsilon_t` |
| **Estimation** | Ordinary least squares | Maximum likelihood |
| **Diagnostics** | Standard regression metrics | AIC/BIC, confidence intervals on `phi_i` |
| **Order selection** | Manual (we pick one lag) | ACF/PACF plot + information criteria |
| **Multiple lags** | Add columns manually | Handled automatically by `lags=p` |

> 🧠 **Key insight:** you are not doing something wrong here. Linear regression on a single lag feature IS a valid time-series model. The AR framework gives you additional statistical machinery (coefficient tests, order selection criteria, proper forecast intervals) — but the underlying math is the same. The predictions will look similar; the diagnostic toolkit is richer.

> ❗️ **But wait — the coefficient interpretation shifts.** In house price regression, `beta_lag_1 = 0.95` would mean "a unit increase in lag_1 increases the price by 0.95". Here, `beta_1 = 0.95` means "if today's PM2.5 is 1 unit above the series mean, tomorrow's is expected to be 0.95 units above the mean — slightly pulled back toward average". That small pull toward the mean is **mean reversion**, and it explains why the model outperforms the naive "tomorrow = today" baseline.

### Split

**Code Task 3.2.1.5**: Split the data into training and testing sets using an 80/20 chronological split. The first 80% of observations should be training data, and the last 20% should be testing data. Do NOT use `train_test_split` — manually split using indexing.

> **Chronological Split**
>
> In time series, we **cannot** use random splitting. We must respect time ordering:
> - Training data = Past (earlier observations)
> - Testing data = Future (later observations)
>
> This simulates real-world forecasting: you train on everything you knew up to a cutoff date, then evaluate how well the model predicts what happened after that date.


In [ ]:
from IPython.display import VimeoVideo

# Bigger video
VimeoVideo("1169845196", h="3298dbabb7", width=700, height=450)

In [ ]:
# Calculate split index (80% for training)
split_idx = int(len(X) * 0.8)

# Split chronologically
X_train = X.iloc[:...]
X_test = X.iloc[...:]
y_train = y.iloc[:...]
y_test = y.iloc[...:]

print(f"Training set: {len(X_train)} observations ({len(X_train)/len(X)*100:.1f}%)")
print(f"Testing set: {len(X_test)} observations ({len(X_test)/len(X)*100:.1f}%)")
print(f"\nTraining period: {X_train.index.min()} to {X_train.index.max()}")
print(f"Testing period: {X_test.index.min()} to {X_test.index.max()}")

## 2. Build Model

### Baseline

Before building a more sophisticated model, we need a **baseline** — the simplest possible prediction that we're trying to beat. In time series, the gold-standard naïve baseline is the **persistence model**: predict that tomorrow's value equals today's.

> 💡 **Why the persistence model is the baseline, not zero:** in regression tasks with cross-sectional data, predicting the mean of `y` is the natural baseline (it minimises MSE under no-feature conditions). In time series, predicting `y_{t-1}` for `y_t` is more appropriate — and harder to beat — because the series has memory. Yesterday's PM2.5 is the single best predictor of today's, and any model that fails to beat persistence is not adding value.

> 🎯 **The bar to beat:** if our linear regression model does not clearly outperform the persistence baseline in MAE, we have learned nothing beyond "tomorrow looks like today" — and we paid a model-fitting cost to get that non-result.

The persistence forecast for the test set is simply the lag-1 values: `y_pred_baseline = test_set.shift(1)`. The first predicted value is NaN (no prior observation) and is dropped. Note that this is equivalent to asking: "what would MAE be if our model was literally `y_hat_t = y_{t-1}`?" — the answer tells you how much free information is in the raw autocorrelation of the series.

**Code Task 3.2.2.1**: Create a baseline prediction for the test set where each prediction equals the previous observation (persistence model). Calculate the Mean Absolute Error (MAE) and assign it to `mae_baseline`.


In [ ]:
from sklearn.metrics import mean_absolute_error

# Baseline: predict using the most recent lag (lag_1)
y_pred_baseline = X_test["lag_1"]

# Calculate baseline MAE
mae_baseline = mean_absolute_error(y_test, ...)

print(f"Baseline MAE: {mae_baseline:.2f}")
print(f"\nInterpretation: On average, our baseline predictions are off by {mae_baseline:.2f} PM2.5 units.")

### Iterate

> 💡 **sklearn `LinearRegression` — the fit/predict pattern you know, applied to time series**
>
> ```python
> from sklearn.linear_model import LinearRegression
>
> model = LinearRegression()       # Instantiate (no training yet)
> model.fit(X_train, y_train)      # Learn coefficients on TRAINING data only
>
> y_pred_train = model.predict(X_train)   # In-sample predictions
> y_pred_test  = model.predict(X_test)    # Out-of-sample predictions
> ```
>
> The API is identical to Projects 1 and 2. What changes is the meaning:
> - `X_train` contains only past observations (before the split date)
> - `X_test` contains observations from the future (after the split date)
> - `y_pred_test` is the model's forecast of what it has never seen
>
> The coefficient `model.coef_[0]` is the learned `beta_1` — how much today's PM2.5 changes when yesterday's increases by 1 unit. The intercept `model.intercept_` is `beta_0` — the baseline level when `lag_1 = 0`.


**Code Task 3.2.2.2**: Instantiate a `LinearRegression` model named
`model_lr` and fit it to the training data (`X_train` and `y_train`).

In [ ]:
from sklearn.linear_model import LinearRegression

# Instantiate the model
model_lr = ...

# Fit the model
model_lr.fit(..., ...)

print("✅ Model trained successfully!")
print(f"\nModel coefficients: {model_lr.coef_}")
print(f"Model intercept: {model_lr.intercept_:.2f}")

> **sklearn LinearRegression vs statsmodels OLS**
>
> We’re using `LinearRegression` from **scikit-learn**. But there’s
> another popular implementation: **statsmodels OLS** (Ordinary Least
> Squares). Both do linear regression, but with different strengths:
>
> | Feature | sklearn `LinearRegression` | statsmodels `OLS` |
> |--------------|----------------------------------|-------------------------|
> | **Best for** | Prediction, ML pipelines | Statistical analysis, inference |
> | **Output** | Coefficients, predictions | Detailed summary (p-values, R², confidence intervals) |
> | **Speed** | Optimized for large-scale prediction | More diagnostics, slower |
> | **API** | Simple: `.fit()`, `.predict()` | Statistical: needs formula interface |
> | **Regularization** | Easy to add (Ridge, Lasso) | Separate functions for regularization |
>
> **Why we chose sklearn:**
> - ✅ Students already know it from Project 2
> - ✅ Works seamlessly with sklearn metrics (MAE, RMSE)
> - ✅ Clean API perfect for notebooks
> - ✅ Leads naturally to more advanced sklearn models
>
> **When to use statsmodels:** - Need p-values to test if coefficients
> are significant - Academic research requiring rigorous statistical
> inference - Time series models (AR, ARMA) - which we’ll see in
> Notebooks 3-4!
>
> **Example comparison:**
>
> ``` python
> # sklearn way (what we're using)
> from sklearn.linear_model import LinearRegression
> model = LinearRegression()
> model.fit(X_train, y_train)
> predictions = model.predict(X_test)
>
> # statsmodels way (for comparison)
> import statsmodels.api as sm
> X_train_sm = sm.add_constant(X_train)  # Add intercept
> model_sm = sm.OLS(y_train, X_train_sm).fit()
> print(model_sm.summary())  # Detailed statistical output
> ```
>
> Both give the same mathematical result, but different interfaces for
> different purposes!

**Bonus: What if we used multiple lags?**

Our current model uses only `lag_1`. But what if PM2.5 depends on not
just yesterday’s value, but also the day before, or even the week
before? We could create `lag_2`, `lag_3`, etc., and fit a more complex
model.

Here’s what that would look like:

In [ ]:
# Create multiple lag features (bonus: not in the main task)
df_multiple_lags = (
    df.loc["2024"].copy()
    .assign(lag_1=lambda x: x["pm25"].shift(1))
    .assign(lag_2=lambda x: x["pm25"].shift(2))
    .assign(lag_3=lambda x: x["pm25"].shift(3))
    .assign(lag_7=lambda x: x["pm25"].shift(7))  # <-- one week ago!
    .dropna()
)

# Fit a model with all lags
X_multi = df_multiple_lags[["lag_1", "lag_2", "lag_3", "lag_7"]]
y_multi = df_multiple_lags["pm25"]

model_multi = LinearRegression()
model_multi.fit(X_multi, y_multi)

# Compare coefficients
print("Coefficients for different lags:")
for lag, coef in zip(["lag_1", "lag_2", "lag_3", "lag_7"], model_multi.coef_):
    print(f"  {lag:8s}: {coef:7.4f}")

**What you’d notice:** - `lag_1` typically has the largest coefficient
(recent history matters most) - `lag_2` and `lag_3` have smaller
coefficients (diminishing effect as you go back) - `lag_7` (one week
ago) might have a notable coefficient if there are weekly patterns

This is the **intuition** behind autoregressive AR(p) models we’ll build
in Lesson 3!

**Code Task 3.2.2.3**: Use your fitted model to generate predictions for
both `X_train` and `X_test`. Assign them to `y_pred_train` and
`y_pred_test` respectively.

In [ ]:
# Generate predictions
y_pred_train = model_lr.predict(...)
y_pred_test = model_lr.predict(...)

print(f"Training predictions shape: {y_pred_train.shape}")
print(f"Testing predictions shape: {y_pred_test.shape}")
print(f"\nFirst 5 test predictions: {y_pred_test[:5]}")

> 🔍 **Before the scatter plot — what to look for:**
> - In the **scatter plot** (actual vs predicted), points close to the diagonal line indicate good predictions. Points above the line mean the model underestimates; below means overestimates. Systematic patterns in the scatter (a fan shape, a curve) indicate heteroscedasticity or non-linearity.
> - In the **time series plot** (actual and predicted over time), look for whether the predicted line **lags the actual by one step** — this "echo" pattern is characteristic of lag-1 models and shows that the model's forecast is essentially the previous observation, not much improved.


### Evaluate

> 💡 **Three metrics, one story:**
>
> | Metric | Formula | Units | Key question |
> |--------|---------|-------|-------------|
> | **MAE** | mean(abs(y - y_hat)) | µg/m³ | Average absolute miss |
> | **RMSE** | sqrt(mean((y - y_hat)²)) | µg/m³ | Same as MAE, but penalises large errors more |
> | **R²** | 1 - SS_res/SS_tot | dimensionless [0,1] | Fraction of variance explained |
>
> **In a time-series context, MAE is the most interpretable:** "on average, our prediction is X µg/m³ from the truth." It's in the same units as PM2.5, so you can directly answer "how wrong are we, on average?"
>
> **RMSE penalises large misses disproportionately.** If your model has an MAE of 8 µg/m³ but occasionally misses by 40 µg/m³ (during a spike), RMSE will be much higher than MAE. A large MAE/RMSE gap tells you the model has trouble with extremes.
>
> **R² can be misleading** for autocorrelated series. A high R² does not necessarily mean the model is good — it may simply reflect that yesterday's value is a strong predictor (which the persistence model captures equally well). Always compare R² *between models* rather than reading it in isolation.
>
> **The real test:** compare MAE (linear regression) against MAE (persistence baseline). The percentage reduction in MAE over the baseline is the cleanest measure of what the model actually adds beyond "tomorrow = today."


**Code Task 3.2.2.4**: Calculate three evaluation metrics—MAE, RMSE, and
R²—for both the training and testing sets. Also calculate RMSE for the
baseline for comparison. Store them in a DataFrame named `metrics_df`
for easy comparison.

In [ ]:
import numpy as np
from sklearn.metrics import mean_squared_error, r2_score

# Calculate metrics for training set
mae_train = mean_absolute_error(y_train, ...)
rmse_train = np.sqrt(mean_squared_error(y_train, ...))
r2_train = r2_score(y_train, ...)

# Calculate metrics for testing set
mae_test = mean_absolute_error(y_test, ...)
rmse_test = np.sqrt(mean_squared_error(y_test, ...))
r2_test = r2_score(y_test, ...)

# Calculate baseline RMSE for comparison
rmse_baseline = np.sqrt(mean_squared_error(y_test, ...))  # <-- baseline predictions

# Create comparison DataFrame
metrics_df = pd.DataFrame({
    "Metric": ["MAE", "RMSE", "R²"],
    "Training": [mae_train, rmse_train, r2_train],
    "Testing": [mae_test, rmse_test, r2_test],
    "Baseline": [mae_baseline, rmse_baseline, None]
})

print(metrics_df.round(2))

print(f"\n📊 Key Insight:")
print(f"   - Baseline MAE: {mae_baseline:.2f}")
print(f"   - Model Test MAE: {mae_test:.2f}")
print(f"   - Improvement: {((mae_baseline - mae_test) / mae_baseline * 100):.1f}% better than baseline")

**Code 3.2.2.1**: Create a scatter plot comparing actual vs predicted
values for both training and test sets. Include a diagonal line
representing perfect predictions.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 5))

# Training set
axes[0].scatter(y_train, y_pred_train, alpha=0.5)
axes[0].plot([y_train.min(), y_train.max()], 
             [y_train.min(), y_train.max()], 
             'r--', lw=2, label='Perfect Prediction')
axes[0].set_xlabel('Actual PM2.5')
axes[0].set_ylabel('Predicted PM2.5')
axes[0].set_title(f'Training Set (R² = {r2_train:.3f})')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Test set
axes[1].scatter(y_test, y_pred_test, alpha=0.5, color='green')
axes[1].plot([y_test.min(), y_test.max()], 
             [y_test.min(), y_test.max()], 
             'r--', lw=2, label='Perfect Prediction')
axes[1].set_xlabel('Actual PM2.5')
axes[1].set_ylabel('Predicted PM2.5')
axes[1].set_title(f'Test Set (R² = {r2_test:.3f})')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

**Code 3.2.2.2**: Create a time series plot showing actual vs predicted
values for the test set only.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))

# Plot actual values
ax.plot(y_test.index, y_test, label='Actual', alpha=0.7)

# Plot predicted values
ax.plot(y_test.index, y_pred_test, label='Predicted', alpha=0.7)

ax.set_xlabel('Time')
ax.set_ylabel('PM2.5 Concentration')
ax.set_title('Actual vs Predicted PM2.5 (Test Set)')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

> 📊 **Interpreting the actual vs predicted plot:**
> - Does the predicted line **shadow the actual line one step behind**? That "one-step echo" means the model is essentially doing persistence — predicting tomorrow from today without much improvement.
> - Do the predicted values **underestimate spikes**? AR-style models with a coefficient slightly below 1 systematically pull extreme predictions toward the mean — this is mean reversion at work. The model is right to do this on average, even if individual spikes are underestimated.
> - Does the model track **trend changes** quickly? If PM2.5 starts rising sharply, a lag-1 model adjusts one step at a time — it will lag behind. This is an inherent limitation of single-lag linear models.
>
> **Key distinction — scatter vs time-series plot:**
> - The **scatter plot** (actual vs predicted) is good for spotting systematic bias (if points cluster above or below the diagonal, the model is systematically over- or under-predicting).
> - The **time series plot** (both series plotted against the same time axis) is good for spotting timing issues — if the predicted curve looks like a slightly-shifted version of the actual curve, that's the signature of a pure lag-1 model with near-unit coefficient.


In [ ]:
# Get feature names and coefficients
feature_names = X.columns
coefficients = model_lr.coef_

# Create DataFrame for plotting
importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Coefficient': coefficients
}).sort_values('Coefficient', ascending=False)

print("Feature Importance (Coefficients):")
print(importance_df)

# Create bar plot
fig, ax = plt.subplots(figsize=(10, 6))
ax.bar(importance_df['Feature'], importance_df['Coefficient'])
ax.set_xlabel('Lag Feature')
ax.set_ylabel('Coefficient Value')
ax.set_title('Linear Regression Coefficients by Lag Feature')
ax.grid(True, alpha=0.3, axis='y')
ax.axhline(y=0, color='red', linestyle='--', linewidth=1)

plt.tight_layout()
plt.show()

> 📊 **Reading the coefficients:**
> - The `lag_1` coefficient is typically close to — but slightly below — 1.0. A value of `0.95` means: "for every 1 µg/m³ of PM2.5 yesterday, today is expected to be 0.95 µg/m³." The 0.05 pull toward the mean is **mean reversion**.
> - The intercept is the model's baseline. When `lag_1 = 0` (theoretically), the model predicts this value — but since PM2.5 is never zero, the intercept is best interpreted as a term that shifts the overall level up or down.
> - If you fitted multiple lags (the bonus section), notice that `lag_1` dominates and subsequent lags have diminishing coefficients — recent history matters most.
> - **A coefficient above 1.0 would be a warning sign** — it would imply explosive dynamics: a 1-unit increase today leads to a > 1-unit increase tomorrow, leading to exponential growth in the long run. Real PM2.5 doesn't behave that way, so a coefficient near 0.9–0.98 is realistic and expected.


## Summary

You have built your first time series forecasting model — a linear regression trained on a single lag feature — and evaluated it against the persistence baseline.

### What You Built

| Step | What you did | Why it matters |
|------|--------------|----------------|
| Visualised PM2.5 | Time series plot | Understand patterns before modelling |
| Created `lag_1` | `df["pm25"].shift(1)` | Core autoregressive feature |
| Split chronologically | 80% past / 20% future | Prevents future leakage into training |
| Set persistence baseline | Predict `y_{t-1}` for `y_t` | The minimum bar any model must clear |
| Fit `LinearRegression` | `model.fit(X_train, y_train)` | Learns `beta_0 + beta_1 * lag_1` |
| Evaluated with 3 metrics | MAE, RMSE, R² | Complete picture of forecast quality |

### Key Insights

- **Temporal dependence invalidates i.i.d. assumptions.** PM2.5 at time `t` is not independent of PM2.5 at time `t-1`. The whole modelling approach must change: no random shuffling, no random cross-validation, no treating rows as exchangeable.
- **The persistence baseline is harder to beat than it looks.** Because PM2.5 is strongly autocorrelated, yesterday's value is already a very good predictor of today's. Any model that fails to clearly beat it should be scrapped, not polished.
- **Mean reversion explains the `beta_1 < 1` coefficient.** A coefficient of 0.95 is not a failure to learn the identity function. It reflects a genuine property of the data: extreme values pull back toward the long-run mean.

### Interpreting the Coefficient: When the Model Beats the Baseline

The `lag_1` coefficient is typically close to — but slightly below — 1.0. If `beta_1 = 0.95` and `beta_0 = 2.1`, the model predicts:

$$\hat{y}_{t+1} = 2.1 + 0.95 \cdot y_t$$

When `y_t = 80` (a pollution spike), the naive baseline predicts 80. The model predicts `2.1 + 0.95 × 80 = 78.1`. That 1.9 µg/m³ correction is **mean reversion**: the model has learned that very high values tend to come down slightly the next period.

Over many predictions, those 1–2 µg/m³ corrections accumulate into a meaningful MAE improvement over the persistence baseline.

### Limitations and Next Steps

Our single-lag linear model has real limitations:
- **One lag only**: it ignores patterns at lag 2, 3, or 7 (a week ago). Multiple lags could capture weekly seasonality.
- **No external features**: weather (wind speed, humidity) and traffic patterns influence PM2.5 but are not included.
- **Assumes linearity**: the relationship between `y_t` and `y_{t-1}` may not be linear, especially during pollution events.
- **Static model**: once trained, the model's coefficients don't update as new data arrives.

In Lesson 3, we will address the first limitation head-on with **autoregressive models** — a statistical framework specifically designed for time series that automatically handles multiple lags, provides confidence intervals on each coefficient, and offers principled criteria (AIC/BIC) for choosing the right number of lags. The key upgrade: instead of manually picking `lag_1`, the AR(p) framework uses the **ACF and PACF diagnostic plots** to tell you how many lags actually carry predictive signal.

## Multiple Choice Questions

✅ **Now it is time to answer the multiple choice questions.**

------------------------------------------------------------------------

## Discussion Questions

- Why is it important to establish a baseline before building complex
  models?

- Look at the model coefficient for the lag_1 feature. What does this
  coefficient tell you about the relationship between current and past
  PM2.5 values? Does this make intuitive sense?

- In the predicted vs actual scatter plot, do you notice any patterns in
  the residuals (points that deviate from the diagonal line)? What might
  cause these deviations?

- Our model only uses past PM2.5 values. What external factors (weather,
  traffic, industrial activity) might improve predictions if we had that
  data?

- The chronological train/test split simulates real forecasting. In
  practice, how often would you need to retrain your model as new data
  becomes available?

------------------------------------------------------------------------
